In [1]:
import sys
sys.path.insert(1, '../../scripts/')
from core.reaction import Metabolic_Reaction, Expression_Reaction
from macromolecules.complex import Complex

No objective coefficients in model. Unclear what should be optimized


In [61]:
def flatten_list(t):
    #https://stackoverflow.com/questions/952914/how-to-make-a-flat-list-out-of-list-of-lists
    return [item for sublist in t for item in sublist]

class Expressed_Gene():
    '''Tracks all reactions and macromolecules associated with a ME Model gene.
    
    Designed to be used after building the full ME_Model
    '''
    def __init__(self, hgnc_id):
        '''
        Init method
        
        Parameters
        ----------
        hgnc_id: str
            the gene HGNC ID
        
        '''
        if not hgnc_id.startswith('HGNC:'):
            raise ValueuError('Currently all genes must be in standard HGNC ID format')
        self.hgnc_id = hgnc_id
        # initialize reactions
        self.reactions = {'Catalysis_Reactions': {'Metabolic_Module': dict(), 'Expression_Module': dict()}, 
                          'Expression_Reactions': {'mrna': {'synthesis': None, 'sink': None, 'other': []}, 
                                                  'protein': {'translation': [], 'synthesis': [], 
                                                              'sink': [], 'other': []}, 
                                                  'complex': {'synthesis': dict(), 'sink': dict(), 'other': dict()}}}
        self.macromolecules = {'RNA': {'premrna': None, 'mrna': {'coupled': {}, 'other': None}, 'lariat': None}, 
                              'Protein':None,
                              'Complex': None,
                              'Proxy': None
                             }
    # REACTIONS--------------------------------------------------------------------------------
    def add_reaction(self, r):
        '''Organizes ME_Model reaction into self.reactions attribute'''
        catalysis = self.is_catalyzing(r)
        if catalysis:
            self._add_catalysis_reaction(r)
        
        expression = False
        if isinstance(r, Expression_Reaction): # not elif, can be catalyzing its own expression
            expression = self._add_expression_reaction(r)
        
        if not catalysis and not expression:
            raise ValueError('The reaction ' + r.id + ' does not appear to be associated with the gene ' + self.hgnc_id)
   
    def is_catalyzing(self, r):
        '''Determines whether the gene is involved in catalysis of the reaction
        
        Parameters
        ----------
        r: instance of ME_Reaction
        
        Returns
        ----------
        catalysis: bool
            True if the gene is involved in catalyzing the reaction, False otherwise
        
        '''
        
        catalysis = True
        assoc_macro = {t: m for m,t in r.coupled_metabolites.items()}
        if ('catalysis' in assoc_macro):
            if not isinstance(assoc_macro['catalysis'], Complex): # monomers
                for m,t in r.coupled_metabolites.items():
                    if t not in ['catalysis', 'enzyme_degradation']:
                        raise ValueError('Unexpected coupling type in catalysis reaction for ' + r.id)
                    if m.hgnc_id != self.hgnc_id:
                        catalysis = False
            else: # complexes                 
                if self.hgnc_id not in [m.hgnc_id for m in assoc_macro['catalysis'].decompose_complex()]:
                    catalysis = False
        else:
            catalysis = False
        return catalysis
    
    def _add_catalysis_reaction(self, r):
        '''
        Adds reactions that the gene catalyzes, splitting by whether it is catalyzing a metabolic or 
        expression module reaction. 
        
        Hierarchy is organized as follows: {reaction_id: {catalysis: enzyme_id, deg_proxy: proxy_id}}
        
        Where catalysis: enzyme_id represents the macromolecules enzyme that is coupled to the reaction for 
        protein synthesis to reaction catalysis and deg_proxy: proxy_id represents the proxy macromolecule that couples 
        protein degradation to reaction catalysis. 
        
        '''
        
        
        if isinstance(r, Metabolic_Reaction):
            self.reactions['Catalysis_Reactions']['Metabolic_Module'][r.id] = {t: m.id for m,t in r.coupled_metabolites.items()}
        else:
            self.reactions['Catalysis_Reactions']['Expression_Module'][r.id] = {t: m.id for m,t in r.coupled_metabolites.items()}
    
    
    def _add_expression_reaction(self, r, tol = 1e-17):
        '''
        Reactions involving expression of a gene (not catalysis, even if it is catalysis of an expression-module reaction)
        
        Parameters
        ----------
        tol: float
            tolerance threshold for determining whether a complex is self-catalysing the reaction (exceptional cases)
        '''
        expression = True
                              
        # mrna
        if r.subsystem == 'mRNA_expression':  
            if r.hgnc_id != self.hgnc_id:
                expression = False
            
            else:
                if r.synthesis:
                    if self.reactions['Expression_Reactions']['mrna']['synthesis'] != None:
                        raise ValueError('Multiple ' + 'mrna' + '  synthesis reactions assigned to ' + self.hgnc_id)
                    self.reactions['Expression_Reactions']['mrna']['synthesis'] = r.id
                elif r.sink:
                    if self.reactions['Expression_Reactions']['mrna']['sink'] is not None:
                        raise ValueError('Multiple ' + 'mrna' + '  sink reactions assigned to ' + self.hgnc_id)
                    self.reactions['Expression_Reactions']['mrna']['sink'] = r.id
                else:
                    self.reactions['Expression_Reactions']['mrna']['other'] += [r.id]                                           
            
        # protein
        # multiple compartments allow multiple sink/synthesis reactions
        elif r.subsystem.startswith('Protein_'):
            if r.hgnc_id != self.hgnc_id:
                expression = False
            else:    
                if r.synthesis:
                    self.reactions['Expression_Reactions']['protein']['synthesis'] += [r.id]
                elif r.sink:
                    self.reactions['Expression_Reactions']['protein']['sink'] += [r.id]
                elif hasattr(r, 'translation') and r.translation:
                    if len(self.reactions['Expression_Reactions']['protein']['translation']) > 1:
                        raise ValueError('More than two ' + 'protein' + '  translation reactions assigned to ' + self.hgnc_id)
                    self.reactions['Expression_Reactions']['protein']['translation'] += [r.id]
                else:
                    self.reactions['Expression_Reactions']['protein']['other'] += [r.id]
                                                   
        # complex
        elif r.subsystem.startswith('Complex_'): 
            complexes = list()
            for cplx in r.metabolites:
                if isinstance(cplx, Complex):
                    if self.hgnc_id in [m.hgnc_id for m in cplx.decompose_complex()]:
                        if cplx in r.coupled_metabolites: # account for self-catalysis 
                            # note that the two internal abs() shouldn't need to be there, but in case of 
                            # inconsistent formatting of coupling coefficient values
                            if abs(r.metabolites[cplx] - cplx.coupling_coefficient[r.coupled_metabolites[cplx]])>tol:
                                complexes += [cplx]
                        else:
                            complexes += [cplx]
#             complexes = [cplx for cplx in r.products if isinstance(cplx, Complex) and cplx not in r.coupled_metabolites \
#                         and self.hgnc_id in [m.hgnc_id for m in cplx.components]]
            if len(complexes) == 0:
                expression = False
            else:
                if r.synthesis:
                    complex_product = [c for c in complexes if c in r.products]
                    if len(complex_product) > 1:
                        raise ValueError('Unexpected synthesis of multiple complex products')
                    complex_product = complex_product[0]
                    if complex_product.id in self.reactions['Expression_Reactions']['complex']['synthesis']:
                        raise ValueError('Multiple ' + 'complex' + '  synthesis reactions assigned to ' + complex_product.id)
                        
                    self.reactions['Expression_Reactions']['complex']['synthesis'][complex_product.id] = r.id
                    for cplx in set(complexes).difference([complex_product]):
                        self._add_other_complex_expression(cplx, r)
                elif r.sink:
                    complex_reactant = [c for c in complexes if c in r.reactants]
                    if len(complex_reactant) > 1:
                        raise ValueError('Unexpected degradation of multiple complex reactants')
                    complex_reactant = complex_reactant[0]
                    if complex_reactant.id in self.reactions['Expression_Reactions']['complex']['sink']:
                        raise ValueError('Multiple ' + 'complex' + '  sink reactions assigned to ' + complex_reactant.id)

                    self.reactions['Expression_Reactions']['complex']['sink'][complex_reactant.id] = r.id
                    for cplx in set(complexes).difference([complex_reactant]):
                        self._add_other_complex_expression(cplx, r)
                else:
                    self._add_other_complex_expression(cplx, r)
        else:
            raise ValueError('Unaccounted for Expression Reaction subsystem')
        return expression
    
    def _add_other_complex_expression(self, cplx, r):
        if cplx.id not in self.reactions['Expression_Reactions']['complex']['other']:
            self.reactions['Expression_Reactions']['complex']['other'][cplx.id] = [r.id]
        else: 
            self.reactions['Expression_Reactions']['complex']['other'][cplx.id] += [r.id]
                                                   
    def check_reactions(self):
        if len(self.reactions['Catalysis_Reactions']['Metabolic_Module']) == 0 and len(self.reactions['Catalysis_Reactions']['Expression_Module']) == 0:
            raise ValueError('No catalysis reactions associated with this gene')
        mrna = self.reactions['Expression_Reactions']['mrna']
        protein = self.reactions['Expression_Reactions']['protein']
        complex_ = self.reactions['Expression_Reactions']['complex']
        
        
        
        if mrna['synthesis'] is None:
            raise ValueError('No mrna synthesis reactions associated with this gene')
        if mrna['sink'] is None:
            raise ValueError('No mrna degradation reactions associated with this gene')
        if len(protein['translation']) == 0:
            raise ValueError('No protein translation reactions associated with this gene')
        if len(protein['synthesis']) == 0 and len(complex_['synthesis']) == 0:
            raise ValueError('No enzyme synthesis reactions associated with this gene')
        if len(protein['sink']) == 0 and len(complex_['sink']) == 0:
            raise ValueError('No enzyme degradation reactions associated with this gene')
        
        # all proteins have atleast 1 translation reaction
        # appropriate couplings between reactions
    # MACROMOLECULES--------------------------------------------------------------------------------
    def add_macromolecule(self, m):
        if not hasattr(m, 'hgnc_id') or m.hgnc_id is None or m.hgnc_id != self.hgnc_id:
            raise ValueError('The macromolecule ' + m.id + ' does not appear to be associated with the gene ' + self.hgnc_id)
        if m.type == 'fragment_rna':
            if self.macromolecules['lariat'] is not None:
                raise ValueError('Multiple ' + 'lariats' + ' assigned to ' + self.hgnc_id)
            self.macromolecules['RNA']['lariat'] = m.id
        elif m.type == 'premrna':
            if self.macromolecules['premrna'] is not None:
                raise ValueError('Multiple ' + 'premrna' + ' assigned to ' + self.hgnc_id)
            self.macromolecules['RNA']['premrna'] = m.id 
        elif m.type == 'mrna':
            if m.coupling_coefficient is None: 
                if m.id is not None:
                    raise ValueError('Multiple uncoupled' + 'mrna' + ' assigned to ' + self.hgnc_id)
                self.macromolecules['RNA']['mrna']['other'] = m.id
            else:
                if m.id in self.macromolecules['RNA']['mrna']['coupled']:
                    raise ValueError('Multiple reactions coupled to' + 'mrna' + ' for ' + self.hgnc_id)
                else:
                    if len(self.macromolecules['RNA']['mrna']['coupled']) > 0:
                        raise ValueError('Multiple coupled' + 'mrna' + ' assigned to ' + self.hgnc_id)
                    if list(m.coupling_coefficient.keys()) != ['mrna_formation']:
                        raise ValueError('Unexpected coupling type for mrna associated with ' + self.hgnc_id)

                    cr = [r for r in m.reactions if m in r.coupled_metabolites]

                    if len(cr) > 2:
                        raise ValueError('Unexpected mrna coupling to multiple reactions associated with ' + self.hgnc_id)
                    self.macromolecules['RNA']['mrna']['coupled'][m.id] = cr

In [4]:
import pickle
import sys
sys.path.insert(1, '../../scripts/')
from core.reaction import Metabolic_Reaction, Expression_Reaction
from core.model import load_pickled_model
from macromolecules.macromolecule import Macromolecule
from macromolecules.protein import Protein

lp_path = '/data2/hratch/human_me/other/test_lp/'

me_model = load_pickled_model(lp_path + 'working_version_' + str(4) + '.pickle')





In [5]:
macromolecules = [m for m in me_model.metabolites if isinstance(m, Macromolecule)]

In [6]:
set([m.type for m in macromolecules])

{'complex',
 'dummy_protein',
 'fragment_rna',
 'mrna',
 'protein',
 'proxy',
 'rrna',
 'trna'}

In [62]:
macromolecules = [m for m in macromolecules if m.type in ['fragment_rna', 'premrna', 'mrna'] \
                  and m.hgnc_id is not None]

In [63]:
for m in macromolecules:
    g = Expressed_Gene(m.hgnc_id)
    g.add_macromolecule(m)

In [56]:
g = Expressed_Gene(m.hgnc_id)
# g.add_macromolecule(m)

In [57]:
m

Metabolite identifier,HGNC:643_mrna_c
Name,
Memory address,0x07f1bfd4eb160
Formula,C28036H31630N10991O20600P2951
Compartment,c
In 4 reaction(s),"HGNC:643_DECAPPING_mRNA_DEGRADATIONc, HGNC:643_TRANSCRIPTION, HGNC:643_co_TRANSLOC_IMPORTtr, HGNC:643_TRANSLATION_ELONGATIONc"


In [58]:
self = copy.deepcopy(g)

In [51]:
g.macromolecules

{'RNA': {'premrna': None,
  'mrna': {'coupled': {}, 'other': None},
  'lariat': None},
 'Protein': None,
 'Complex': None,
 'Proxy': None}

In [59]:
cr = [r for r in m.reactions if m in r.coupled_metabolites]



In [60]:
cr

[<Protein_Expression_Reaction HGNC:643_co_TRANSLOC_IMPORTtr at 0x7f1bfd507320>,
 <Protein_Expression_Reaction HGNC:643_TRANSLATION_ELONGATIONc at 0x7f1bfd4eb470>]

In [50]:
if m.type == 'mrna':
    if m.coupling_coefficient is None: 
        if m.id is not None:
            raise ValueError('Multiple uncoupled' + 'mrna' + ' assigned to ' + self.hgnc_id)
        self.macromolecules['RNA']['mrna']['other'] = m.id
    else:
        if m.id in self.macromolecules['RNA']['mrna']['coupled']:
            raise ValueError('Multiple reactions coupled to' + 'mrna' + ' for ' + self.hgnc_id)
        else:
            if len(self.macromolecules['RNA']['mrna']['coupled']) > 0:
                raise ValueError('Multiple coupled' + 'mrna' + ' assigned to ' + self.hgnc_id)
            if list(m.coupling_coefficient.keys()) != ['mrna_formation']:
                raise ValueError('Unexpected coupling type for mrna associated with ' + self.hgnc_id)

            cr = [r for r in m.reactions if m in r.coupled_metabolites]

            if len(cr) > 0:
                raise ValueError('Unexpected mrna coupling to multiple reactions associated with ' + self.hgnc_id)
            self.macromolecules['RNA']['mrna']['coupled'][m.id] = cr[0].id

ValueError: Unexpected mrna coupling to multiple reactions associated with HGNC:10404

In [42]:
if m.id in self.macromolecules['RNA']['mrna']['coupled']:
    raise ValueError('Multiple reactions coupled to' + 'mrna' + ' for ' + self.hgnc_id)
else:
    if len(self.macromolecules['RNA']['mrna']['coupled']) > 0:
        raise ValueError('Multiple coupled' + 'mrna' + ' assigned to ' + self.hgnc_id)
    if list(m.coupling_coefficient.keys()) != ['mrna_formation']:
        raise ValueError('Unexpected coupling type for mrna associated with ' + self.hgnc_id)



ValueError: Unexpected mrna coupling to multiple reactions associated with HGNC:10404

In [33]:
g.macromolecules
    

AttributeError: 'Expressed_Gene' object has no attribute 'macromolecules'

In [10]:
self.macromlecules = {'RNA': {'premrna': None, 'mrna': {'coupled': {}, 'other': None}, 'lariat': None}, 
                              'Protein':None,
                              'Complex': None,
                              'Proxy': None
                             }

{'mrna_formation': -(mu + 0.0227449473351396)/(98855.3094656939*mu + 957.985789455595)}

[<Protein_Expression_Reaction HGNC:10404_TRANSLATION_ELONGATIONc at 0x7f1bfd7e2eb8>]

In [20]:
list(m.reactions)[0].coupled_metabolites

{<Complex TRANSCRIPTION_complex_n at 0x7f1bfd8719b0>: 'catalysis',
 <Macromolecule TRANSCRIPTION_COMPLEX_enzyme_deg_proxy_n at 0x7f1bfd7e2e80>: 'enzyme_degradation'}

There is lack of updating of objects throughout. I have a method that updates the metabolite objects to include all the reactions they are involved in. After which, we can see something like this:

In [ ]:
self.macromlecules = {'RNA': {'premrna': None, 'mrna': {'coupled': {}, 'other': None}, 'lariat': None}, 
                              'Protein':None,
                              'Complex': None,
                              'Proxy': None
                             }

In [56]:
me_model._map_metabolite_reactions()

In [ ]:
self.macromlecules = {'RNA': {'premrna': None, 'mrna': {'coupled': None, 'other': None}, 'lariat': None}
                              'Protein':
                              'Complex': 
                              'Proxy': 
                             }

# test reactions

In [ ]:
# hgnc_id = 'HGNC:24723' # monomer catalyzes metabolic reaction

# reactions = list(set(flatten_list([[r for r in list(m.reactions)] for m in me_model.metabolites if isinstance(m, Macromolecule) and m.hgnc_id == hgnc_id])))
# g = Expressed_Gene(hgnc_id)
# for r in reactions:
#     print(r.id)
#     g.add_reaction(r)
# g.check_reactions() 

# hgnc_id = 'HGNC:2698' # complex catalyzes metabolic reaction

# all_metabs = list()
# for m in me_model.metabolites:
#     if isinstance(m, Complex):
#         for m_ in m.decompose_complex():
#             if isinstance(m_, Protein) and m_.hgnc_id == hgnc_id:
#                 all_metabs += [cplx.id, m_.id]
#     else:
#         if hasattr(m, 'hgnc_id') and m.hgnc_id == hgnc_id:
#             all_metabs += [m.id]
# all_metabs = set(all_metabs)
# reactions = [me_model.reactions.get_by_id(r_id) for r_id in list(set(flatten_list([[r.id for r in me_model.metabolites.get_by_id(m_id).reactions] for m_id in all_metabs])))]
            
# g = Expressed_Gene(hgnc_id)
# for r in reactions:
#     print(r.id)
#     g.add_reaction(r)
# g.check_reactions()

# hgnc_id = 'HGNC:10404' # ribosomal protein

# all_metabs = list()
# for m in me_model.metabolites:
#     if isinstance(m, Complex):
#         for m_ in m.decompose_complex():
#             if isinstance(m_, Protein) and m_.hgnc_id == hgnc_id:
#                 all_metabs += [m.id, m_.id]
#     else:
#         if hasattr(m, 'hgnc_id') and m.hgnc_id == hgnc_id:
#             all_metabs += [m.id]
# all_metabs = set(all_metabs)
# reactions = [me_model.reactions.get_by_id(r_id) for r_id in list(set(flatten_list([[r.id for r in me_model.metabolites.get_by_id(m_id).reactions] for m_id in all_metabs])))]

# g = Expressed_Gene(hgnc_id)
# for r in reactions:
#     print(r.id)
#     g.add_reaction(r)